# 🦀 Day 1: Unsafe Rust — Raw Pointers & Unsafe Blocks
---

Today we explore **unsafe Rust** — the escape hatch when the compiler can't prove safety.

- **What unsafe Rust is**: The 5 unsafe superpowers
- **Raw pointers**: `*const T` and `*mut T`
- **Creating vs dereferencing**: Safe to create, unsafe to dereference
- **`unsafe {}` blocks**: Explicitly opting into responsibility
- **When unsafe is needed**: FFI, performance-critical code, hardware access
- **The unsafe contract**: What you promise the compiler
- **Safe abstractions**: Wrapping unsafe code in safe APIs

In [ ]:
// Raw pointers: *const T (immutable) and *mut T (mutable)
// Creating raw pointers from references is SAFE
let x = 42i32;
let ptr: *const i32 = &x;
let mut y = 100i32;
let mut_ptr: *mut i32 = &mut y;

println!("ptr address: {:p}", ptr);
println!("mut_ptr address: {:p}", mut_ptr);
// Dereferencing requires unsafe — we'll do that next

In [ ]:
// Dereferencing raw pointers is UNSAFE — we must use unsafe {}
let x = 42i32;
let ptr: *const i32 = &x;

unsafe {
    let value = *ptr;
    println!("Dereferenced value: {}", value);
}

// The 5 unsafe superpowers (only allowed inside unsafe):
// 1. Dereference raw pointers
// 2. Call unsafe functions
// 3. Access or modify mutable statics
// 4. Implement unsafe traits
// 5. Access fields of unions

In [ ]:
// Accessing mutable statics requires unsafe
static mut COUNTER: u32 = 0;

fn increment_counter() {
    unsafe {
        COUNTER += 1;
    }
}

increment_counter();
increment_counter();
unsafe {
    println!("Counter: {}", COUNTER);
}

In [ ]:
// Safe abstraction: split_at_mut — split a slice into two mutable halves
// (std::slice has this; we implement it to show the pattern)
fn split_at_mut<T>(slice: &mut [T], mid: usize) -> (&mut [T], &mut [T]) {
    let len = slice.len();
    let ptr = slice.as_mut_ptr();
    assert!(mid <= len);
    unsafe {
        (
            std::slice::from_raw_parts_mut(ptr, mid),
            std::slice::from_raw_parts_mut(ptr.add(mid), len - mid),
        )
    }
}

let mut v = vec![1, 2, 3, 4, 5];
let (left, right) = split_at_mut(&mut v[..], 3);
left[0] = 100;
right[0] = 200;
println!("left: {:?}, right: {:?}", left, right);

## 📝 Exercise: Safe wrapper around raw pointer

Implement a safe function `read_at<T>(ptr: *const T) -> Option<T>` that:
- Returns `Some(value)` if the pointer is non-null and valid
- Returns `None` if the pointer is null
- Uses `unsafe` only where necessary and documents the safety invariants

In [ ]:
// YOUR CODE HERE
// fn read_at<T>(ptr: *const T) -> Option<T> {
//     if ptr.is_null() { return None; }
//     unsafe { Some(std::ptr::read(ptr)) }
// }
// let x = 42;
// println!("{:?}", read_at(&x));
// println!("{:?}", read_at(std::ptr::null()));

## 🎯 Key Takeaways

1. **Raw pointers** `*const T` and `*mut T` can be created from references (safe)
2. **Dereferencing** raw pointers requires `unsafe { *ptr }`
3. The **5 unsafe superpowers**: deref raw ptrs, call unsafe fn, mutable statics, unsafe traits, union fields
4. **`unsafe`** doesn't disable checks — it means *you* guarantee safety
5. **Safe abstractions** wrap unsafe code so callers never need `unsafe`

---
**Tomorrow:** Unsafe functions and traits